# 06 — Walk-Forward Backtest
Simulate the full trading strategy using walk-forward test predictions.
Includes trade management (partial TPs, breakeven stops, trailing).

In [ ]:
# Install dependencies
# torch, numpy, pandas are pre-installed by Colab — do NOT reinstall them.
# Pinning versions fights Colab's environment and causes resolver conflicts.
!pip install -q xgboost ccxt PyWavelets hmmlearn numba scikit-learn pyyaml \
    tqdm pyarrow matplotlib seaborn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json

# Clone/update repo from GitHub (local Colab filesystem — fast)
REPO_DIR = '/content/scalp2_repo'
if os.path.exists(os.path.join(REPO_DIR, '.git')):
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone https://github.com/sergul74/Scalp2.git {REPO_DIR}

if not os.path.exists(os.path.join(REPO_DIR, 'scalp2', '__init__.py')):
    raise FileNotFoundError('scalp2 package not found after clone!')

sys.path.insert(0, REPO_DIR)

import logging
logging.basicConfig(level=logging.INFO)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scalp2.config import load_config
from scalp2.execution.trade_manager import TradeManager, TradeState, TradeStatus
from scalp2.utils.metrics import sharpe_ratio, sortino_ratio, max_drawdown, win_rate, profit_factor

config = load_config(f'{REPO_DIR}/config.yaml')
DATA_DIR = '/content/drive/MyDrive/scalp2/data/processed'

In [ ]:
# Load labeled dataset and walk-forward predictions
import pickle

df = pd.read_parquet(f'{DATA_DIR}/BTC_USDT_labeled.parquet')

with open(f'{DATA_DIR}/wf_predictions.pkl', 'rb') as f:
    wf_predictions = pickle.load(f)

print(f'Loaded {len(df)} bars, {len(wf_predictions)} walk-forward folds')

In [ ]:
# Walk-forward backtest with trade management + realistic costs + regime filters
# Fixes applied:
#   1.2  Entry at NEXT bar open (not signal bar close)
#   1.3  Choppy regime filter using forward-only HMM probs
#   1.4  Fractional Kelly position sizing (matches live)
#   1.6  Multi-leg slippage model (per-exit cost)
#   1.7  Leverage, variable slippage, funding rate, market impact
from tqdm import tqdm

# Suppress per-trade log spam from TradeManager
logging.getLogger('scalp2.execution.trade_manager').setLevel(logging.WARNING)

seq_len = config.model.seq_len
exec_cfg = config.execution
trade_mgmt_cfg = config.execution.trade_management
label_cfg = config.labeling
order_cfg = config.execution.order_execution

trade_mgr = TradeManager(trade_mgmt_cfg, label_cfg.max_holding_bars)

# --- Transaction costs from config (Binance USDM Futures) ---
MAKER_FEE_BPS = order_cfg.maker_fee_bps
TAKER_FEE_BPS = order_cfg.taker_fee_bps
SLIPPAGE_BPS = order_cfg.slippage_bps

# --- Leverage & margin ---
LEVERAGE = exec_cfg.position_sizing.leverage
MAINT_MARGIN = exec_cfg.position_sizing.maintenance_margin

# --- Variable slippage model ---
slip_cfg = getattr(exec_cfg, 'slippage_model', None)
USE_VAR_SLIPPAGE = slip_cfg is not None and slip_cfg.enabled
if USE_VAR_SLIPPAGE and 'atr_14' in df.columns:
    MEDIAN_ATR = df['atr_14'].median()
else:
    MEDIAN_ATR = 1.0

def get_slippage_bps(atr_val):
    """Variable slippage: base + volatility component."""
    if not USE_VAR_SLIPPAGE:
        return SLIPPAGE_BPS
    ratio = atr_val / (MEDIAN_ATR + 1e-10)
    return slip_cfg.base_bps + slip_cfg.volatility_scale * ratio

# --- Funding rate model ---
funding_cfg = getattr(exec_cfg, 'funding_rate', None)
USE_FUNDING = funding_cfg is not None and funding_cfg.enabled

def count_funding_intervals(entry_time, exit_time):
    """Count 8h funding intervals between entry and exit."""
    if not USE_FUNDING:
        return 0
    funding_times = pd.date_range(
        entry_time.normalize(),
        exit_time.normalize() + pd.Timedelta(days=1),
        freq='8h'
    )
    return int(((funding_times > entry_time) & (funding_times <= exit_time)).sum())

# --- Market impact model ---
impact_cfg = getattr(exec_cfg, 'market_impact', None)
USE_MARKET_IMPACT = impact_cfg is not None and impact_cfg.enabled

def market_impact_frac(position_size, price):
    """Proportional market impact for large positions (returned as fraction)."""
    if not USE_MARKET_IMPACT:
        return 0.0
    notional = position_size * price * LEVERAGE
    return impact_cfg.base_impact_bps * (notional / impact_cfg.reference_notional_usd) / 10_000

# Per-leg cost functions (in fraction, not bps)
def entry_cost_frac(slip_bps):
    """Entry is a limit order: maker fee + slippage."""
    return (MAKER_FEE_BPS + slip_bps) / 10_000

def exit_cost_frac(status, slip_bps):
    """Exit cost depends on how the trade closed.
    SL/time/regime exits are market orders (taker fee).
    TP exits are limit orders (maker fee).
    """
    if status in (TradeStatus.CLOSED_SL, TradeStatus.CLOSED_TIME,
                  TradeStatus.CLOSED_REGIME, 'CLOSED_FOLD_END'):
        return (TAKER_FEE_BPS + slip_bps) / 10_000
    else:  # CLOSED_TP
        return (MAKER_FEE_BPS + slip_bps) / 10_000

# Legacy flat cost for comparison
FLAT_RT_COST = 2 * (SLIPPAGE_BPS + MAKER_FEE_BPS) / 10_000
print(f'TX cost model: multi-leg | maker={MAKER_FEE_BPS}bps taker={TAKER_FEE_BPS}bps '
      f'slip={SLIPPAGE_BPS}bps | flat RT={FLAT_RT_COST*10000:.0f}bps (for reference)')
print(f'Leverage: {LEVERAGE}x | Variable slippage: {USE_VAR_SLIPPAGE} | '
      f'Funding rate: {USE_FUNDING} | Market impact: {USE_MARKET_IMPACT}')

# --- Kelly sizing params ---
# Effective b-ratio accounts for partial TP
partial_pct = trade_mgmt_cfg.partial_tp_1_pct   # 0.5
partial_atr = trade_mgmt_cfg.partial_tp_1_atr   # 0.6
full_tp_atr = trade_mgmt_cfg.full_tp_atr        # 1.2
effective_tp = partial_pct * partial_atr + (1 - partial_pct) * full_tp_atr
kelly_b = effective_tp / label_cfg.sl_multiplier
kelly_fraction = exec_cfg.position_sizing.kelly_fraction   # 0.25
kelly_max = exec_cfg.position_sizing.max_fraction          # 0.02
print(f'Kelly sizing: effective_b={kelly_b:.2f} (was {label_cfg.tp_multiplier/label_cfg.sl_multiplier:.2f}), '
      f'fraction={kelly_fraction}, max={kelly_max}')

# --- Precompute ADX and ATR percentile for filtering ---
MIN_ADX = exec_cfg.min_adx
MIN_ATR_PCTILE = exec_cfg.min_atr_percentile
CHOPPY_THRESHOLD = config.regime.choppy_threshold

# Rolling ATR percentile (96-bar window = ~24h on 15m)
if 'atr_14' in df.columns:
    df['atr_pctile'] = df['atr_14'].rolling(96, min_periods=10).rank(pct=True)
    df['atr_pctile'] = df['atr_pctile'].fillna(1.0)
else:
    df['atr_pctile'] = 1.0

print(f'Filters: min_adx={MIN_ADX}, min_atr_pctile={MIN_ATR_PCTILE}, '
      f'choppy_threshold={CHOPPY_THRESHOLD}')

all_trades = []
equity_curve = [0.0]
cumulative_pnl = 0.0
skip_reasons = {'low_adx': 0, 'low_volatility': 0, 'low_conf': 0,
                'hold': 0, 'daily_cap': 0, 'no_atr': 0, 'choppy': 0,
                'no_next_bar': 0}
liquidated = False

def _close_trade(active, position_size, entry_bar, bar, row, fold_idx,
                 status_override=None, gross_override=None):
    """Record a closed trade with all cost layers and leverage."""
    global cumulative_pnl

    gross = gross_override if gross_override is not None else active.pnl
    status_val = status_override if status_override else active.status.value

    # Variable slippage based on ATR at entry
    slip_bps = get_slippage_bps(active.atr_at_entry)

    # Multi-leg cost
    cost = entry_cost_frac(slip_bps)
    if active.partial_fills:
        cost += exit_cost_frac(TradeStatus.CLOSED_TP, slip_bps) * partial_pct
        exit_status = status_override or active.status
        cost += exit_cost_frac(exit_status, slip_bps) * (1 - partial_pct)
    else:
        exit_status = status_override or active.status
        cost += exit_cost_frac(exit_status, slip_bps)

    # Market impact
    impact = market_impact_frac(position_size, active.entry_price)
    cost += impact

    # Funding rate
    entry_ts = df.index[entry_bar]
    exit_ts = row.name
    n_funding = count_funding_intervals(entry_ts, exit_ts)
    funding = n_funding * (funding_cfg.fixed_rate_pct / 100.0) if USE_FUNDING else 0.0

    # Per-unit net (unleveraged)
    unit_net = (gross - cost - funding) * position_size

    # Leveraged portfolio PnL
    leveraged_net = unit_net * LEVERAGE

    cumulative_pnl += leveraged_net

    all_trades.append(dict(
        fold=fold_idx, direction=active.direction,
        entry_price=active.entry_price,
        bars_held=active.bars_held,
        status=status_val,
        gross_pnl=gross * position_size * LEVERAGE,
        unit_pnl=unit_net,
        net_pnl=leveraged_net,
        cost=cost * position_size * LEVERAGE,
        funding_cost=funding * position_size * LEVERAGE,
        position_size=position_size,
        margin_utilization=position_size * LEVERAGE,
        slippage_bps=slip_bps,
        impact_bps=impact * 10_000,
        n_exits=1 + len(active.partial_fills),
        entry_bar=entry_bar, exit_bar=bar,
        timestamp=row.name,
    ))
    equity_curve.append(cumulative_pnl)

for fold_data in tqdm(wf_predictions, desc='Backtesting folds'):
    if liquidated:
        break

    fold_idx = fold_data['fold_idx']
    test_start = fold_data['test_start']
    test_end = fold_data['test_end']
    preds = fold_data['test_probabilities']   # (n_preds, 3)
    n_preds = len(preds)
    pred_offset = test_start + seq_len        # first bar with a prediction

    # Regime probs (forward-only, from stage2_trainer)
    # Fallback: if not present (old wf_predictions), skip regime filter
    regime_probs = fold_data.get('regime_probs', None)
    has_regime = regime_probs is not None

    active = None
    entry_bar = None
    pending_signal = None   # (direction, atr, sl_raw, confidence, pred_idx)
    daily_count = 0
    prev_date = None

    for i in range(n_preds):
        bar = pred_offset + i
        if bar >= len(df):
            break

        row = df.iloc[bar]
        cur_date = row.name.date() if hasattr(row.name, 'date') else None
        if cur_date != prev_date:
            daily_count = 0
            prev_date = cur_date

        # ---- manage active trade ----
        if active is not None:
            # Check choppy regime for active trade
            is_choppy = False
            if has_regime and i < len(regime_probs):
                is_choppy = regime_probs[i, 2] > CHOPPY_THRESHOLD

            active = trade_mgr.update(
                active, row['high'], row['low'], row['close'], is_choppy
            )
            if active.status not in (TradeStatus.OPEN, TradeStatus.PARTIAL_TP):
                _close_trade(active, position_size, entry_bar, bar, row, fold_idx)

                # Liquidation check
                if cumulative_pnl <= -1.0:
                    print(f'LIQUIDATED at fold {fold_idx}, bar {bar}')
                    liquidated = True
                    break

                active = None
            continue  # don't open new trade on same bar

        # ---- execute pending signal from previous bar ----
        if pending_signal is not None:
            ps = pending_signal
            pending_signal = None

            entry_price = row['open']  # NEXT BAR OPEN (realistic fill)
            atr = ps['atr']
            direction = ps['direction']
            confidence = ps['confidence']

            # Recompute SL from actual entry price
            if direction == "LONG":
                sl = entry_price - label_cfg.sl_multiplier * atr
            else:
                sl = entry_price + label_cfg.sl_multiplier * atr

            # Kelly position sizing
            p = confidence
            q = 1 - p
            kelly = max((p * kelly_b - q) / kelly_b, 0)
            position_size = min(kelly * kelly_fraction, kelly_max)

            if position_size < 1e-6:
                continue  # skip if Kelly says don't trade

            active = TradeState(
                direction=direction,
                entry_price=entry_price,
                current_stop_loss=sl,
                take_profit=0.0,
                atr_at_entry=atr,
            )
            entry_bar = bar
            daily_count += 1
            continue  # don't evaluate signal on entry bar

        # ---- check for new signal ----
        p = preds[i]
        cls = int(np.argmax(p))
        if cls == 1:
            skip_reasons['hold'] += 1
            continue
        if max(p[0], p[2]) < exec_cfg.confidence_threshold:
            skip_reasons['low_conf'] += 1
            continue
        if daily_count >= exec_cfg.max_trades_per_day:
            skip_reasons['daily_cap'] += 1
            continue

        atr = row['atr_14'] if 'atr_14' in df.columns else 0.0
        if atr < 1e-10:
            skip_reasons['no_atr'] += 1
            continue

        # ADX filter
        adx_val = row.get('adx_14', 999.0) if hasattr(row, 'get') else (
            row['adx_14'] if 'adx_14' in df.columns else 999.0)
        if adx_val < MIN_ADX:
            skip_reasons['low_adx'] += 1
            continue

        # ATR percentile filter
        atr_pct = row['atr_pctile'] if 'atr_pctile' in df.columns else 1.0
        if atr_pct < MIN_ATR_PCTILE:
            skip_reasons['low_volatility'] += 1
            continue

        # Choppy regime filter (forward-only probs)
        if has_regime and i < len(regime_probs):
            if regime_probs[i, 2] > CHOPPY_THRESHOLD:
                skip_reasons['choppy'] += 1
                continue

        # Check next bar exists for entry
        next_bar = pred_offset + i + 1
        if next_bar >= len(df):
            skip_reasons['no_next_bar'] += 1
            continue

        direction = "LONG" if cls == 2 else "SHORT"

        # Set pending signal — will execute at next bar open
        pending_signal = {
            'direction': direction,
            'atr': atr,
            'confidence': max(p[0], p[2]),
        }

    # force-close any open trade at fold boundary
    if active is not None:
        last_row = df.iloc[min(test_end - 1, len(df) - 1)]
        if active.direction == "LONG":
            unr = (last_row['close'] - active.entry_price) / active.entry_price
        else:
            unr = (active.entry_price - last_row['close']) / active.entry_price
        gross = active.pnl + unr * active.remaining_size

        _close_trade(active, position_size, entry_bar,
                     min(test_end - 1, len(df) - 1), last_row, fold_idx,
                     status_override='CLOSED_FOLD_END', gross_override=gross)
        active = None
    # Clear pending signal at fold boundary
    pending_signal = None

trades_df = pd.DataFrame(all_trades)
print(f'\nTotal trades: {len(trades_df)}')
print(f'Cumulative net PnL ({LEVERAGE}x levered): {cumulative_pnl*100:.2f}%')
print(f'\nSkip reasons: {skip_reasons}')
if len(trades_df) > 0:
    print(f'Avg position size: {trades_df["position_size"].mean():.4f}')
    print(f'Avg margin util: {trades_df["margin_utilization"].mean()*100:.1f}%')
    print(f'Avg cost/trade (levered): {trades_df["cost"].mean()*10000:.1f} bps')
    print(f'Regime filter active: {has_regime}')
    if liquidated:
        print('*** ACCOUNT LIQUIDATED ***')

In [ ]:
# Results summary and visualization
if len(trades_df) == 0:
    print("No trades generated!")
else:
    net = trades_df['net_pnl'].values
    gross = trades_df['gross_pnl'].values
    unit = trades_df['unit_pnl'].values
    n = len(trades_df)

    # Win/loss stats (on leveraged net)
    wins = net[net > 0]
    losses = net[net < 0]
    wr = len(wins) / n
    avg_win = wins.mean() if len(wins) else 0
    avg_loss = losses.mean() if len(losses) else 0
    pf = abs(wins.sum() / losses.sum()) if len(losses) else float('inf')

    # Daily P&L for Sharpe/Sortino (leveraged)
    trades_df['date'] = pd.to_datetime(trades_df['timestamp']).dt.date
    daily_pnl = trades_df.groupby('date')['net_pnl'].sum()
    full_range = pd.date_range(daily_pnl.index.min(), daily_pnl.index.max(), freq='D')
    daily_pnl = daily_pnl.reindex(full_range, fill_value=0.0)

    daily_sharpe = daily_pnl.mean() / (daily_pnl.std() + 1e-10) * np.sqrt(365)
    down = daily_pnl[daily_pnl < 0]
    daily_sortino = daily_pnl.mean() / (down.std() + 1e-10) * np.sqrt(365) if len(down) else 0

    # Max drawdown (on leveraged equity)
    cum = np.cumsum(daily_pnl.values)
    peak = np.maximum.accumulate(cum)
    dd = peak - cum
    mdd = dd.max()
    calmar = (daily_pnl.mean() * 365) / mdd if mdd > 1e-10 else 0

    # Status distribution
    status_counts = trades_df['status'].value_counts()

    # Per-unit (unleveraged) stats for comparison
    unit_daily = trades_df.groupby('date')['unit_pnl'].sum().reindex(full_range, fill_value=0.0)
    unit_cum = np.cumsum(unit_daily.values)
    unit_peak = np.maximum.accumulate(unit_cum)
    unit_dd = unit_peak - unit_cum
    unit_mdd = unit_dd.max()

    print('=' * 60)
    print('       WALK-FORWARD BACKTEST RESULTS')
    print(f'       (Leverage: {LEVERAGE}x)')
    print('=' * 60)
    print(f'  Total trades       : {n}')
    print(f'  Win rate           : {wr:.4f} ({wr*100:.1f}%)')
    print(f'  Profit factor      : {pf:.4f}')
    print(f'  Expectancy/trade   : {net.mean()*100:.4f}%')
    print(f'  Avg win            : +{avg_win*100:.4f}%')
    print(f'  Avg loss           : {avg_loss*100:.4f}%')
    print(f'  Avg bars held      : {trades_df["bars_held"].mean():.1f}')
    print()
    print(f'  Daily Sharpe       : {daily_sharpe:.4f}')
    print(f'  Daily Sortino      : {daily_sortino:.4f}')
    print(f'  Max Drawdown       : {mdd*100:.4f}%')
    print(f'  Calmar             : {calmar:.4f}')
    print()
    print(f'  Gross PnL          : {gross.sum()*100:.2f}%')
    print(f'  Net PnL            : {net.sum()*100:.2f}%')
    print(f'  Cost impact        : {(gross.sum()-net.sum())*100:.2f}%')
    print(f'  TX cost/trade      : {FLAT_RT_COST*10000:.0f} bps (flat ref)')
    print()
    print('  Close reasons:')
    for status, count in status_counts.items():
        print(f'    {status:20s}: {count:5d} ({count/n*100:.1f}%)')

    # --- Leverage & cost model analysis ---
    print()
    print('-' * 60)
    print('  LEVERAGE & COST MODEL ANALYSIS')
    print('-' * 60)
    print(f'  Leverage            : {LEVERAGE}x')
    print(f'  Net PnL (per-unit)  : {unit.sum()*100:.2f}%')
    print(f'  Net PnL (leveraged) : {net.sum()*100:.2f}%')
    print(f'  MDD (per-unit)      : {unit_mdd*100:.4f}%')
    print(f'  MDD (leveraged)     : {mdd*100:.4f}%')
    print(f'  Avg position size   : {trades_df["position_size"].mean()*100:.2f}%')
    print(f'  Avg margin util     : {trades_df["margin_utilization"].mean()*100:.1f}%')
    print(f'  Max margin util     : {trades_df["margin_utilization"].max()*100:.1f}%')
    if USE_FUNDING:
        total_funding = trades_df['funding_cost'].sum()
        print(f'  Total funding cost  : {total_funding*100:.4f}%')
        n_funded = (trades_df['funding_cost'] > 0).sum()
        print(f'  Trades w/ funding   : {n_funded} ({n_funded/n*100:.1f}%)')
    if USE_VAR_SLIPPAGE:
        print(f'  Avg slippage        : {trades_df["slippage_bps"].mean():.1f} bps')
        print(f'  Max slippage        : {trades_df["slippage_bps"].max():.1f} bps')
    if USE_MARKET_IMPACT:
        print(f'  Avg market impact   : {trades_df["impact_bps"].mean():.2f} bps')
        print(f'  Max market impact   : {trades_df["impact_bps"].max():.2f} bps')
    if liquidated:
        print(f'  *** ACCOUNT LIQUIDATED ***')
    print('=' * 60)

    # --- Charts ---
    fig, axes = plt.subplots(3, 1, figsize=(16, 12))

    # 1. Equity curve (leveraged)
    cum_pct = np.cumsum(daily_pnl.values) * 100
    axes[0].plot(daily_pnl.index, cum_pct, linewidth=1, color='#2196F3')
    axes[0].fill_between(daily_pnl.index, 0, cum_pct, alpha=0.1, color='#2196F3')
    axes[0].axhline(0, color='grey', linestyle='--', alpha=0.5)
    axes[0].set_title(f'Equity Curve (Cumulative PnL % — {LEVERAGE}x Leveraged)')
    axes[0].set_ylabel('Cumulative PnL (%)')
    axes[0].grid(True, alpha=0.3)

    # 2. Drawdown (leveraged)
    axes[1].fill_between(daily_pnl.index, 0, -dd * 100, alpha=0.4, color='red')
    axes[1].set_title(f'Drawdown ({LEVERAGE}x Leveraged)')
    axes[1].set_ylabel('Drawdown (%)')
    axes[1].grid(True, alpha=0.3)

    # 3. PnL distribution
    axes[2].hist(net * 100, bins=50, alpha=0.7, color='#4CAF50', edgecolor='white')
    axes[2].axvline(0, color='red', linestyle='--', alpha=0.7)
    axes[2].set_title('Trade PnL Distribution (Leveraged)')
    axes[2].set_xlabel('PnL per trade (%)')
    axes[2].set_ylabel('Count')
    axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    # Monthly returns table (leveraged)
    trades_df['month'] = pd.to_datetime(trades_df['timestamp']).dt.to_period('M')
    monthly = trades_df.groupby('month')['net_pnl'].agg(['sum', 'count'])
    monthly.columns = ['return_pct', 'trades']
    monthly['return_pct'] *= 100
    print('\nMonthly Returns:')
    print(monthly.to_string())

# Pre-Sample Out-of-Sample Test (2020)
Test Fold 0's model on **2020 data** — a year the model has never seen during training.
Training starts at 2021-01-01, so 2020 is completely out-of-sample.

If the model has real predictive edge → WR > 55%, PF > 1.5
If overfitting → WR ≈ 33%, PF ≈ 1.0

In [ ]:
# ============================================================
# OOS Step 1: Download 2020 data & build features
# ============================================================
import torch
import copy
from scalp2.data.preprocessing import clean_ohlcv, resample_ohlcv
from scalp2.features.builder import build_features, drop_warmup_nans, get_feature_columns
from scalp2.data.mtf_builder import build_mtf_dataset

# Download 2020 data using ccxt directly (avoid Binance region blocks)
import ccxt

# Try multiple exchanges until one works
EXCHANGES_TO_TRY = [
    ('binanceus', 'BTC/USDT'),
    ('bybit', 'BTC/USDT'),
    ('okx', 'BTC/USDT'),
    ('kraken', 'BTC/USD'),
]

import os
cache_path = '/content/drive/MyDrive/scalp2/data/processed/btc_2020_15m.parquet'

if os.path.exists(cache_path):
    print(f"Loading cached 2020 data from {cache_path}")
    df_2020_raw = pd.read_parquet(cache_path)
    if 'timestamp' in df_2020_raw.columns:
        df_2020_raw['timestamp'] = pd.to_datetime(df_2020_raw['timestamp'])
        df_2020_raw = df_2020_raw.set_index('timestamp')
    if df_2020_raw.index.tz is not None:
        df_2020_raw.index = df_2020_raw.index.tz_localize(None)
else:
    df_2020_raw = None
    for exch_id, symbol in EXCHANGES_TO_TRY:
        print(f"Trying {exch_id} ({symbol})...")
        try:
            exchange = getattr(ccxt, exch_id)({'enableRateLimit': True})
            start_ms = exchange.parse8601('2020-01-01T00:00:00Z')
            end_ms = exchange.parse8601('2020-12-31T23:59:59Z')

            all_candles = []
            since = start_ms
            while since < end_ms:
                candles = exchange.fetch_ohlcv(symbol, '15m', since=since, limit=1000)
                if not candles:
                    break
                all_candles.extend(candles)
                since = candles[-1][0] + 1
                if len(all_candles) % 10000 < 1000:
                    print(f"  {len(all_candles)} candles...")
                import time; time.sleep(exchange.rateLimit / 1000.0)

            if len(all_candles) > 1000:
                df_2020_raw = pd.DataFrame(all_candles,
                    columns=['timestamp', 'open', 'high', 'low', 'close', 'volume'])
                df_2020_raw['timestamp'] = pd.to_datetime(df_2020_raw['timestamp'], unit='ms')
                df_2020_raw = df_2020_raw.drop_duplicates('timestamp').sort_values('timestamp')
                df_2020_raw = df_2020_raw.set_index('timestamp')
                for col in ['open', 'high', 'low', 'close', 'volume']:
                    df_2020_raw[col] = df_2020_raw[col].astype(np.float32)
                # Cache to Drive
                df_2020_raw.to_parquet(cache_path)
                print(f"SUCCESS: {exch_id} — {len(df_2020_raw)} bars downloaded & cached")
                break
            else:
                print(f"  {exch_id}: too few candles ({len(all_candles)}), skipping")
        except Exception as e:
            print(f"  {exch_id} failed: {e}")
            continue

    if df_2020_raw is None:
        raise RuntimeError("All exchanges failed. Try uploading 2020 data manually.")

print(f"2020 data: {len(df_2020_raw)} bars, {df_2020_raw.index[0]} to {df_2020_raw.index[-1]}")

# Clean and resample to 1H/4H
df_2020_15m = clean_ohlcv(df_2020_raw, '15m')
df_2020_1h = resample_ohlcv(df_2020_15m, '1h')
df_2020_4h = resample_ohlcv(df_2020_15m, '4h')

print(f"Clean 15m: {len(df_2020_15m)}, 1H: {len(df_2020_1h)}, 4H: {len(df_2020_4h)}")

# Build features for each timeframe
print("Building features for 2020...")
df_2020_15m_feat = build_features(df_2020_15m, config.features)
df_2020_1h_feat = build_features(df_2020_1h, config.features)
df_2020_4h_feat = build_features(df_2020_4h, config.features)

# Merge MTF features onto 15m
df_2020_full = build_mtf_dataset(df_2020_15m_feat, df_2020_1h_feat, df_2020_4h_feat)
df_2020_full = drop_warmup_nans(df_2020_full)

oos_feature_cols = get_feature_columns(df_2020_full)
print(f"2020 feature matrix: {len(df_2020_full)} rows x {len(oos_feature_cols)} features")

In [ ]:
# ============================================================
# OOS Step 2: Load Fold 0 artifacts & run inference on 2020
# ============================================================
from scalp2.utils.serialization import load_fold_artifacts
from scalp2.models.hybrid import HybridEncoder
from scalp2.models.meta_learner import XGBoostMetaLearner
from scalp2.data.dataset import ScalpDataset
from torch.utils.data import DataLoader
from torch.amp import autocast

CHECKPOINT_DIR = '/content/drive/MyDrive/scalp2/data/processed'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load Fold 0 artifacts: scaler, HMM, feature indices
print("Loading Fold 0 artifacts...")
fold0 = load_fold_artifacts(CHECKPOINT_DIR, fold_idx=0, device=device)

scaler_0 = fold0['scaler']
top_indices_0 = fold0['top_feature_indices']
feature_names_0 = fold0['feature_names']
regime_detector_0 = fold0.get('regime_detector', None)

# Handle empty top_feature_indices — use first top_k features as fallback
top_k = config.model.handcrafted_top_k
if len(top_indices_0) == 0:
    print(f"  WARNING: top_feature_indices is empty, using first {top_k} features as fallback")
    top_indices_0 = np.arange(min(top_k, len(feature_names_0)), dtype=np.intp)

print(f"  Scaler: {type(scaler_0).__name__}")
print(f"  Top features: {len(top_indices_0)}")
print(f"  Regime detector: {'loaded' if regime_detector_0 else 'not found'}")

# --- Align 2020 features to Fold 0's feature set ---
train_feature_cols = feature_names_0

# Find matching columns between 2020 and Fold 0
missing_cols = [c for c in train_feature_cols if c not in df_2020_full.columns]
if missing_cols:
    print(f"  WARNING: {len(missing_cols)} features missing in 2020 data (likely orderflow).")
    print(f"  Missing: {missing_cols[:10]}...")
    for col in missing_cols:
        df_2020_full[col] = 0.0

# Extract features in training order and scale with Fold 0's scaler
oos_features_raw = df_2020_full[train_feature_cols].values.astype(np.float32)
oos_features_scaled = scaler_0.transform(oos_features_raw).astype(np.float32)
oos_features_scaled = np.nan_to_num(oos_features_scaled, nan=0.0, posinf=0.0, neginf=0.0)

print(f"  Scaled features shape: {oos_features_scaled.shape}")

# --- Reconstruct Fold 0's HybridEncoder ---
encoder_0 = HybridEncoder(
    n_features=oos_features_scaled.shape[1],
    config=config.model,
).to(device)
encoder_0.load_state_dict(fold0['model_state'])
encoder_0.eval()
print(f"  HybridEncoder loaded: {sum(p.numel() for p in encoder_0.parameters())} params")

# --- Extract latent vectors ---
print("Extracting latent vectors for 2020...")
dummy_labels = np.zeros(len(oos_features_scaled), dtype=np.int64)
dummy_returns = np.zeros(len(oos_features_scaled), dtype=np.float32)

ds = ScalpDataset(oos_features_scaled, dummy_labels, dummy_returns, seq_len)
loader = DataLoader(ds, batch_size=512, shuffle=False, pin_memory=True)

latents_list = []
with torch.no_grad():
    for batch_x, _, _ in loader:
        batch_x = batch_x.to(device)
        with autocast('cuda', enabled=True):
            _, latent = encoder_0(batch_x)
        latents_list.append(latent.cpu().numpy())

oos_latents = np.concatenate(latents_list, axis=0)
print(f"  Latent vectors: {oos_latents.shape}")

# --- Regime probs (forward-only) ---
oos_df_aligned = df_2020_full.iloc[seq_len:]

if regime_detector_0 is not None:
    print("Computing forward-only regime probabilities...")
    oos_regime_probs = regime_detector_0.predict_proba_online(oos_df_aligned)
else:
    print("No regime detector — using uniform priors")
    oos_regime_probs = np.full((len(oos_df_aligned), 3), 1.0 / 3, dtype=np.float32)

# --- Handcrafted features (top-K from Fold 0) ---
oos_handcrafted = oos_features_scaled[seq_len:][:, top_indices_0]

# --- Ensure consistent lengths ---
min_len = min(len(oos_latents), len(oos_handcrafted), len(oos_regime_probs))
oos_latents = oos_latents[:min_len]
oos_handcrafted = oos_handcrafted[:min_len]
oos_regime_probs = oos_regime_probs[:min_len]

# --- Build meta-features & predict ---
oos_meta = XGBoostMetaLearner.build_meta_features(
    oos_latents, oos_handcrafted, oos_regime_probs
)

# Load Fold 0 XGBoost
xgb_0 = XGBoostMetaLearner(config.model.xgboost)
xgb_path = f'{CHECKPOINT_DIR}/xgb_fold_000.json'
xgb_0.load(xgb_path)

oos_probs = xgb_0.predict_proba(oos_meta)
print(f"  OOS predictions: {oos_probs.shape}")
print(f"  Class distribution: Short={np.mean(oos_probs.argmax(1)==0)*100:.1f}%, "
      f"Hold={np.mean(oos_probs.argmax(1)==1)*100:.1f}%, "
      f"Long={np.mean(oos_probs.argmax(1)==2)*100:.1f}%")
print(f"  Avg max confidence: {oos_probs.max(1).mean():.3f}")

In [ ]:
# ============================================================
# OOS Step 3: Backtest 2020 predictions with same trade management
# ============================================================
# Uses identical logic to cell-4 but on 2020 data with Fold 0 predictions.

# Prepare 2020 DataFrame for backtest (aligned to latent offset)
df_oos = df_2020_full.iloc[seq_len:seq_len + min_len].copy()

# Compute ATR percentile for 2020
if 'atr_14' in df_oos.columns:
    df_oos['atr_pctile'] = df_oos['atr_14'].rolling(96, min_periods=10).rank(pct=True)
    df_oos['atr_pctile'] = df_oos['atr_pctile'].fillna(1.0)
    OOS_MEDIAN_ATR = df_oos['atr_14'].median()
else:
    df_oos['atr_pctile'] = 1.0
    OOS_MEDIAN_ATR = 1.0

def oos_get_slippage_bps(atr_val):
    if not USE_VAR_SLIPPAGE:
        return SLIPPAGE_BPS
    ratio = atr_val / (OOS_MEDIAN_ATR + 1e-10)
    return slip_cfg.base_bps + slip_cfg.volatility_scale * ratio

# --- OOS Backtest Loop ---
oos_trades = []
oos_equity = [0.0]
oos_cum_pnl = 0.0
oos_skip = {'low_adx': 0, 'low_volatility': 0, 'low_conf': 0,
            'hold': 0, 'daily_cap': 0, 'no_atr': 0, 'choppy': 0,
            'no_next_bar': 0}

oos_trade_mgr = TradeManager(trade_mgmt_cfg, label_cfg.max_holding_bars)
oos_active = None
oos_entry_bar = None
oos_pending = None
oos_daily_count = 0
oos_prev_date = None

n_oos_bars = len(df_oos)
print(f"Running OOS backtest on {n_oos_bars} bars (2020)...")

for i in tqdm(range(n_oos_bars), desc='OOS 2020'):
    row = df_oos.iloc[i]
    cur_date = row.name.date() if hasattr(row.name, 'date') else None
    if cur_date != oos_prev_date:
        oos_daily_count = 0
        oos_prev_date = cur_date

    # ---- manage active trade ----
    if oos_active is not None:
        is_choppy = oos_regime_probs[i, 2] > CHOPPY_THRESHOLD if i < len(oos_regime_probs) else False

        oos_active = oos_trade_mgr.update(
            oos_active, row['high'], row['low'], row['close'], is_choppy
        )
        if oos_active.status not in (TradeStatus.OPEN, TradeStatus.PARTIAL_TP):
            # Close trade with cost model
            gross = oos_active.pnl
            status_val = oos_active.status.value
            slip_bps = oos_get_slippage_bps(oos_active.atr_at_entry)

            cost = entry_cost_frac(slip_bps)
            if oos_active.partial_fills:
                cost += exit_cost_frac(TradeStatus.CLOSED_TP, slip_bps) * partial_pct
                cost += exit_cost_frac(oos_active.status, slip_bps) * (1 - partial_pct)
            else:
                cost += exit_cost_frac(oos_active.status, slip_bps)

            impact = market_impact_frac(oos_pos_size, oos_active.entry_price)
            cost += impact

            entry_ts = df_oos.index[oos_entry_bar_local]
            exit_ts = row.name
            n_fund = count_funding_intervals(entry_ts, exit_ts)
            funding = n_fund * (funding_cfg.fixed_rate_pct / 100.0) if USE_FUNDING else 0.0

            unit_net = (gross - cost - funding) * oos_pos_size
            lev_net = unit_net * LEVERAGE
            oos_cum_pnl += lev_net

            oos_trades.append(dict(
                direction=oos_active.direction,
                entry_price=oos_active.entry_price,
                bars_held=oos_active.bars_held,
                status=status_val,
                gross_pnl=gross * oos_pos_size * LEVERAGE,
                net_pnl=lev_net,
                cost=cost * oos_pos_size * LEVERAGE,
                position_size=oos_pos_size,
                slippage_bps=slip_bps,
                timestamp=row.name,
            ))
            oos_equity.append(oos_cum_pnl)
            oos_active = None
        continue

    # ---- execute pending signal ----
    if oos_pending is not None:
        ps = oos_pending
        oos_pending = None

        entry_price = row['open']
        atr = ps['atr']
        direction = ps['direction']
        confidence = ps['confidence']

        if direction == "LONG":
            sl = entry_price - label_cfg.sl_multiplier * atr
        else:
            sl = entry_price + label_cfg.sl_multiplier * atr

        p_k = confidence
        kelly = max((p_k * kelly_b - (1 - p_k)) / kelly_b, 0)
        oos_pos_size = min(kelly * kelly_fraction, kelly_max)

        if oos_pos_size < 1e-6:
            continue

        oos_active = TradeState(
            direction=direction,
            entry_price=entry_price,
            current_stop_loss=sl,
            take_profit=0.0,
            atr_at_entry=atr,
        )
        oos_entry_bar_local = i
        oos_daily_count += 1
        continue

    # ---- check for new signal ----
    if i >= len(oos_probs):
        continue

    p = oos_probs[i]
    cls = int(np.argmax(p))
    if cls == 1:
        oos_skip['hold'] += 1
        continue
    if max(p[0], p[2]) < exec_cfg.confidence_threshold:
        oos_skip['low_conf'] += 1
        continue
    if oos_daily_count >= exec_cfg.max_trades_per_day:
        oos_skip['daily_cap'] += 1
        continue

    atr = row['atr_14'] if 'atr_14' in df_oos.columns else 0.0
    if atr < 1e-10:
        oos_skip['no_atr'] += 1
        continue

    adx_val = row['adx_14'] if 'adx_14' in df_oos.columns else 999.0
    if adx_val < MIN_ADX:
        oos_skip['low_adx'] += 1
        continue

    atr_pct = row['atr_pctile'] if 'atr_pctile' in df_oos.columns else 1.0
    if atr_pct < MIN_ATR_PCTILE:
        oos_skip['low_volatility'] += 1
        continue

    if i < len(oos_regime_probs) and oos_regime_probs[i, 2] > CHOPPY_THRESHOLD:
        oos_skip['choppy'] += 1
        continue

    if i + 1 >= n_oos_bars:
        oos_skip['no_next_bar'] += 1
        continue

    direction = "LONG" if cls == 2 else "SHORT"
    oos_pending = {
        'direction': direction,
        'atr': atr,
        'confidence': max(p[0], p[2]),
    }

# Force-close at end
if oos_active is not None:
    last_row = df_oos.iloc[-1]
    if oos_active.direction == "LONG":
        unr = (last_row['close'] - oos_active.entry_price) / oos_active.entry_price
    else:
        unr = (oos_active.entry_price - last_row['close']) / oos_active.entry_price
    gross = oos_active.pnl + unr * oos_active.remaining_size
    slip_bps = oos_get_slippage_bps(oos_active.atr_at_entry)
    cost = entry_cost_frac(slip_bps) + exit_cost_frac('CLOSED_FOLD_END', slip_bps)
    unit_net = (gross - cost) * oos_pos_size
    lev_net = unit_net * LEVERAGE
    oos_cum_pnl += lev_net
    oos_trades.append(dict(
        direction=oos_active.direction, entry_price=oos_active.entry_price,
        bars_held=oos_active.bars_held, status='CLOSED_END',
        gross_pnl=gross * oos_pos_size * LEVERAGE, net_pnl=lev_net,
        cost=cost * oos_pos_size * LEVERAGE, position_size=oos_pos_size,
        slippage_bps=slip_bps, timestamp=last_row.name,
    ))
    oos_equity.append(oos_cum_pnl)

# ============================================================
# OOS Results
# ============================================================
oos_df = pd.DataFrame(oos_trades)

if len(oos_df) == 0:
    print("\n*** NO TRADES generated in 2020 OOS test! ***")
    print(f"Skip reasons: {oos_skip}")
    print("Try lowering confidence_threshold or filters for OOS test.")
else:
    oos_net = oos_df['net_pnl'].values
    oos_n = len(oos_df)
    oos_wins = oos_net[oos_net > 0]
    oos_losses = oos_net[oos_net < 0]
    oos_wr = len(oos_wins) / oos_n
    oos_pf = abs(oos_wins.sum() / oos_losses.sum()) if len(oos_losses) else float('inf')

    oos_df['date'] = pd.to_datetime(oos_df['timestamp']).dt.date
    oos_daily = oos_df.groupby('date')['net_pnl'].sum()
    oos_range = pd.date_range(oos_daily.index.min(), oos_daily.index.max(), freq='D')
    oos_daily = oos_daily.reindex(oos_range, fill_value=0.0)
    oos_sharpe = oos_daily.mean() / (oos_daily.std() + 1e-10) * np.sqrt(365)
    oos_cum = np.cumsum(oos_daily.values)
    oos_peak = np.maximum.accumulate(oos_cum)
    oos_dd = oos_peak - oos_cum
    oos_mdd = oos_dd.max()

    print()
    print('=' * 60)
    print('       PRE-SAMPLE OOS TEST — 2020 (Fold 0 Model)')
    print(f'       (Leverage: {LEVERAGE}x)')
    print('=' * 60)
    print(f'  Total trades       : {oos_n}')
    print(f'  Win rate           : {oos_wr:.4f} ({oos_wr*100:.1f}%)')
    print(f'  Profit factor      : {oos_pf:.4f}')
    print(f'  Expectancy/trade   : {oos_net.mean()*100:.4f}%')
    print(f'  Daily Sharpe       : {oos_sharpe:.4f}')
    print(f'  Max Drawdown       : {oos_mdd*100:.4f}%')
    print(f'  Net PnL            : {oos_net.sum()*100:.2f}%')
    print()
    print(f'  Skip reasons: {oos_skip}')
    print()

    # Interpretation
    print('-' * 60)
    print('  INTERPRETATION')
    print('-' * 60)
    if oos_wr > 0.55 and oos_pf > 1.5:
        print('  STRONG: Model shows real predictive edge on unseen data.')
        print('  The learned patterns generalize beyond training period.')
    elif oos_wr > 0.45 and oos_pf > 1.0:
        print('  MODERATE: Some edge exists but weaker than in-sample.')
        print('  Expected degradation — model may be partially regime-dependent.')
    elif oos_wr > 0.33:
        print('  WEAK: Edge is marginal or non-existent.')
        print('  Model may be overfitting to 2021+ market structure.')
    else:
        print('  NEGATIVE: Model performs worse than random on 2020.')
        print('  Suggests systematic bias — investigate immediately.')
    print('=' * 60)

    # Monthly breakdown
    oos_df['month'] = pd.to_datetime(oos_df['timestamp']).dt.to_period('M')
    oos_monthly = oos_df.groupby('month')['net_pnl'].agg(['sum', 'count'])
    oos_monthly.columns = ['return_pct', 'trades']
    oos_monthly['return_pct'] *= 100
    print('\nOOS Monthly Returns (2020):')
    print(oos_monthly.to_string())

    # Equity curve comparison
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    # OOS equity
    oos_cum_pct = np.cumsum(oos_daily.values) * 100
    axes[0].plot(oos_daily.index, oos_cum_pct, linewidth=1.5, color='#FF5722')
    axes[0].fill_between(oos_daily.index, 0, oos_cum_pct, alpha=0.1, color='#FF5722')
    axes[0].axhline(0, color='grey', linestyle='--', alpha=0.5)
    axes[0].set_title(f'OOS 2020 Equity Curve ({LEVERAGE}x)')
    axes[0].set_ylabel('Cumulative PnL (%)')
    axes[0].grid(True, alpha=0.3)

    # OOS drawdown
    axes[1].fill_between(oos_daily.index, 0, -oos_dd * 100, alpha=0.4, color='red')
    axes[1].set_title(f'OOS 2020 Drawdown ({LEVERAGE}x)')
    axes[1].set_ylabel('Drawdown (%)')
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

# Forward Test (2026-02-21 → Present)
Test the **last fold's model** on data **after** the training period ends (2026-02-20).
This is the most relevant OOS test — recent market conditions with the most up-to-date model.

Uses the same multi-exchange download approach with Drive caching.

In [ ]:
# ============================================================
# Forward Test Step 1: Download recent data & build features
# ============================================================
# Need warmup period for features (256 bars wavelet + seq_len)
# Download from 2026-01-15 onward, but only test on bars after 2026-02-20

import ccxt, os, time as _time
from datetime import datetime

FWD_START = "2026-01-15"  # warmup period
FWD_END = datetime.utcnow().strftime("%Y-%m-%d")  # today
FWD_TEST_CUTOFF = "2026-02-20"  # training data ends here

fwd_cache = '/content/drive/MyDrive/scalp2/data/processed/btc_fwd_15m.parquet'

# Always re-download to get latest data (delete cache if stale)
if os.path.exists(fwd_cache):
    cached_df = pd.read_parquet(fwd_cache)
    if 'timestamp' in cached_df.columns:
        last_ts = pd.to_datetime(cached_df['timestamp']).max()
    else:
        last_ts = cached_df.index.max()
    # Re-download if cache is more than 1 day old
    if (pd.Timestamp.utcnow() - pd.Timestamp(last_ts).tz_localize('UTC')).days > 1:
        print(f"Cache stale (last: {last_ts}), re-downloading...")
        os.remove(fwd_cache)

if os.path.exists(fwd_cache):
    print(f"Loading cached forward data from {fwd_cache}")
    df_fwd_raw = pd.read_parquet(fwd_cache)
    if 'timestamp' in df_fwd_raw.columns:
        df_fwd_raw['timestamp'] = pd.to_datetime(df_fwd_raw['timestamp'])
        df_fwd_raw = df_fwd_raw.set_index('timestamp')
    if df_fwd_raw.index.tz is not None:
        df_fwd_raw.index = df_fwd_raw.index.tz_localize(None)
else:
    df_fwd_raw = None
    for exch_id, symbol in EXCHANGES_TO_TRY:
        print(f"Trying {exch_id} ({symbol})...")
        try:
            exchange = getattr(ccxt, exch_id)({'enableRateLimit': True})
            start_ms = exchange.parse8601(f'{FWD_START}T00:00:00Z')
            end_ms = exchange.parse8601(f'{FWD_END}T23:59:59Z')

            all_candles = []
            since = start_ms
            while since < end_ms:
                candles = exchange.fetch_ohlcv(symbol, '15m', since=since, limit=1000)
                if not candles:
                    break
                all_candles.extend(candles)
                since = candles[-1][0] + 1
                if len(all_candles) % 10000 < 1000:
                    print(f"  {len(all_candles)} candles...")
                _time.sleep(exchange.rateLimit / 1000.0)

            if len(all_candles) > 500:
                df_fwd_raw = pd.DataFrame(all_candles,
                    columns=['timestamp', 'open', 'high', 'low', 'close', 'volume'])
                df_fwd_raw['timestamp'] = pd.to_datetime(df_fwd_raw['timestamp'], unit='ms')
                df_fwd_raw = df_fwd_raw.drop_duplicates('timestamp').sort_values('timestamp')
                df_fwd_raw = df_fwd_raw.set_index('timestamp')
                for col in ['open', 'high', 'low', 'close', 'volume']:
                    df_fwd_raw[col] = df_fwd_raw[col].astype(np.float32)
                df_fwd_raw.to_parquet(fwd_cache)
                print(f"SUCCESS: {exch_id} — {len(df_fwd_raw)} bars downloaded & cached")
                break
        except Exception as e:
            print(f"  {exch_id} failed: {e}")
            continue

    if df_fwd_raw is None:
        raise RuntimeError("All exchanges failed for forward data.")

print(f"Forward data: {len(df_fwd_raw)} bars, {df_fwd_raw.index[0]} to {df_fwd_raw.index[-1]}")

# Clean and resample
df_fwd_15m = clean_ohlcv(df_fwd_raw, '15m')
df_fwd_1h = resample_ohlcv(df_fwd_15m, '1h')
df_fwd_4h = resample_ohlcv(df_fwd_15m, '4h')

print(f"Clean 15m: {len(df_fwd_15m)}, 1H: {len(df_fwd_1h)}, 4H: {len(df_fwd_4h)}")

# Build features
print("Building features for forward test...")
df_fwd_15m_feat = build_features(df_fwd_15m, config.features)
df_fwd_1h_feat = build_features(df_fwd_1h, config.features)
df_fwd_4h_feat = build_features(df_fwd_4h, config.features)

df_fwd_full = build_mtf_dataset(df_fwd_15m_feat, df_fwd_1h_feat, df_fwd_4h_feat)
df_fwd_full = drop_warmup_nans(df_fwd_full)

fwd_feature_cols = get_feature_columns(df_fwd_full)
print(f"Forward feature matrix: {len(df_fwd_full)} rows x {len(fwd_feature_cols)} features")
print(f"Test period: {FWD_TEST_CUTOFF} → {df_fwd_full.index[-1]}")

In [ ]:
# ============================================================
# Forward Test Step 2: Load LAST fold's model & run inference
# ============================================================
last_fold_idx = len(wf_predictions) - 1
print(f"Using Fold {last_fold_idx} (last fold) for forward test...")

fold_last = load_fold_artifacts(CHECKPOINT_DIR, fold_idx=last_fold_idx, device=device)

scaler_last = fold_last['scaler']
top_indices_last = fold_last['top_feature_indices']
feature_names_last = fold_last['feature_names']
regime_detector_last = fold_last.get('regime_detector', None)

if len(top_indices_last) == 0:
    print(f"  WARNING: top_feature_indices empty, using first {top_k} features")
    top_indices_last = np.arange(min(top_k, len(feature_names_last)), dtype=np.intp)

print(f"  Scaler: {type(scaler_last).__name__}")
print(f"  Top features: {len(top_indices_last)}")
print(f"  Regime detector: {'loaded' if regime_detector_last else 'not found'}")

# Align features
fwd_feature_cols_model = feature_names_last
missing = [c for c in fwd_feature_cols_model if c not in df_fwd_full.columns]
if missing:
    print(f"  WARNING: {len(missing)} features missing, zero-filling")
    for col in missing:
        df_fwd_full[col] = 0.0

fwd_raw = df_fwd_full[fwd_feature_cols_model].values.astype(np.float32)
fwd_scaled = scaler_last.transform(fwd_raw).astype(np.float32)
fwd_scaled = np.nan_to_num(fwd_scaled, nan=0.0, posinf=0.0, neginf=0.0)

encoder_last = HybridEncoder(
    n_features=fwd_scaled.shape[1],
    config=config.model,
).to(device)
encoder_last.load_state_dict(fold_last['model_state'])
encoder_last.eval()
print(f"  HybridEncoder loaded: {sum(p.numel() for p in encoder_last.parameters())} params")

# Extract latents
print("Extracting latent vectors...")
dummy_l = np.zeros(len(fwd_scaled), dtype=np.int64)
dummy_r = np.zeros(len(fwd_scaled), dtype=np.float32)
ds_fwd = ScalpDataset(fwd_scaled, dummy_l, dummy_r, seq_len)
loader_fwd = DataLoader(ds_fwd, batch_size=512, shuffle=False, pin_memory=True)

fwd_latents = []
with torch.no_grad():
    for bx, _, _ in loader_fwd:
        bx = bx.to(device)
        with autocast('cuda', enabled=True):
            _, lat = encoder_last(bx)
        fwd_latents.append(lat.cpu().numpy())
fwd_latents = np.concatenate(fwd_latents, axis=0)

# Regime probs (forward-only)
fwd_df_aligned = df_fwd_full.iloc[seq_len:]
if regime_detector_last is not None:
    fwd_regime = regime_detector_last.predict_proba_online(fwd_df_aligned)
else:
    fwd_regime = np.full((len(fwd_df_aligned), 3), 1/3, dtype=np.float32)

fwd_hc = fwd_scaled[seq_len:][:, top_indices_last]

fwd_min = min(len(fwd_latents), len(fwd_hc), len(fwd_regime))
fwd_latents = fwd_latents[:fwd_min]
fwd_hc = fwd_hc[:fwd_min]
fwd_regime = fwd_regime[:fwd_min]

fwd_meta = XGBoostMetaLearner.build_meta_features(fwd_latents, fwd_hc, fwd_regime)

xgb_last = XGBoostMetaLearner(config.model.xgboost)
xgb_last.load(f'{CHECKPOINT_DIR}/xgb_fold_{last_fold_idx:03d}.json')

fwd_probs = xgb_last.predict_proba(fwd_meta)
print(f"  Forward predictions: {fwd_probs.shape}")
print(f"  Class dist: Short={np.mean(fwd_probs.argmax(1)==0)*100:.1f}%, "
      f"Hold={np.mean(fwd_probs.argmax(1)==1)*100:.1f}%, "
      f"Long={np.mean(fwd_probs.argmax(1)==2)*100:.1f}%")
print(f"  Avg max confidence: {fwd_probs.max(1).mean():.3f}")

In [ ]:
# ============================================================
# Forward Test Step 3: Backtest only on bars AFTER training cutoff
# ============================================================
df_fwd_bt = df_fwd_full.iloc[seq_len:seq_len + fwd_min].copy()

# Only test bars after the training data cutoff
cutoff_ts = pd.Timestamp(FWD_TEST_CUTOFF)
if df_fwd_bt.index.tz is not None:
    cutoff_ts = cutoff_ts.tz_localize(df_fwd_bt.index.tz)

# Find the index where forward test actually starts
fwd_test_mask = df_fwd_bt.index > cutoff_ts
fwd_test_start_idx = int(np.argmax(fwd_test_mask)) if fwd_test_mask.any() else len(df_fwd_bt)
print(f"Warmup bars (before cutoff): {fwd_test_start_idx}")
print(f"Forward test bars: {fwd_test_mask.sum()}")
print(f"Forward test period: {df_fwd_bt.index[fwd_test_start_idx]} → {df_fwd_bt.index[-1]}")

# ATR percentile
if 'atr_14' in df_fwd_bt.columns:
    df_fwd_bt['atr_pctile'] = df_fwd_bt['atr_14'].rolling(96, min_periods=10).rank(pct=True)
    df_fwd_bt['atr_pctile'] = df_fwd_bt['atr_pctile'].fillna(1.0)
    FWD_MEDIAN_ATR = df_fwd_bt['atr_14'].median()
else:
    df_fwd_bt['atr_pctile'] = 1.0
    FWD_MEDIAN_ATR = 1.0

def fwd_get_slippage_bps(atr_val):
    if not USE_VAR_SLIPPAGE:
        return SLIPPAGE_BPS
    return slip_cfg.base_bps + slip_cfg.volatility_scale * (atr_val / (FWD_MEDIAN_ATR + 1e-10))

# --- Forward Backtest Loop ---
fwd_trades = []
fwd_equity = [0.0]
fwd_cum_pnl = 0.0
fwd_skip = {'low_adx': 0, 'low_volatility': 0, 'low_conf': 0,
            'hold': 0, 'daily_cap': 0, 'no_atr': 0, 'choppy': 0,
            'no_next_bar': 0, 'warmup': 0}

fwd_trade_mgr = TradeManager(trade_mgmt_cfg, label_cfg.max_holding_bars)
fwd_active = None
fwd_pending = None
fwd_daily_count = 0
fwd_prev_date = None
fwd_pos_size = 0.0
fwd_entry_bar_local = 0

n_fwd_bars = len(df_fwd_bt)

for i in tqdm(range(n_fwd_bars), desc='Forward Test'):
    row = df_fwd_bt.iloc[i]
    cur_date = row.name.date() if hasattr(row.name, 'date') else None
    if cur_date != fwd_prev_date:
        fwd_daily_count = 0
        fwd_prev_date = cur_date

    # Manage active trade
    if fwd_active is not None:
        is_choppy = fwd_regime[i, 2] > CHOPPY_THRESHOLD if i < len(fwd_regime) else False
        fwd_active = fwd_trade_mgr.update(fwd_active, row['high'], row['low'], row['close'], is_choppy)
        if fwd_active.status not in (TradeStatus.OPEN, TradeStatus.PARTIAL_TP):
            gross = fwd_active.pnl
            slip_bps = fwd_get_slippage_bps(fwd_active.atr_at_entry)
            cost = entry_cost_frac(slip_bps)
            if fwd_active.partial_fills:
                cost += exit_cost_frac(TradeStatus.CLOSED_TP, slip_bps) * partial_pct
                cost += exit_cost_frac(fwd_active.status, slip_bps) * (1 - partial_pct)
            else:
                cost += exit_cost_frac(fwd_active.status, slip_bps)
            impact = market_impact_frac(fwd_pos_size, fwd_active.entry_price)
            cost += impact
            entry_ts = df_fwd_bt.index[fwd_entry_bar_local]
            n_fund = count_funding_intervals(entry_ts, row.name)
            funding = n_fund * (funding_cfg.fixed_rate_pct / 100.0) if USE_FUNDING else 0.0
            unit_net = (gross - cost - funding) * fwd_pos_size
            lev_net = unit_net * LEVERAGE
            # Only count trades that ENTERED after cutoff
            if fwd_entry_bar_local >= fwd_test_start_idx:
                fwd_cum_pnl += lev_net
                fwd_trades.append(dict(
                    direction=fwd_active.direction, entry_price=fwd_active.entry_price,
                    bars_held=fwd_active.bars_held, status=fwd_active.status.value,
                    gross_pnl=gross * fwd_pos_size * LEVERAGE, net_pnl=lev_net,
                    cost=cost * fwd_pos_size * LEVERAGE, position_size=fwd_pos_size,
                    slippage_bps=slip_bps, timestamp=row.name,
                ))
                fwd_equity.append(fwd_cum_pnl)
            fwd_active = None
        continue

    # Execute pending signal
    if fwd_pending is not None:
        ps = fwd_pending
        fwd_pending = None
        entry_price = row['open']
        atr = ps['atr']
        direction = ps['direction']
        confidence = ps['confidence']
        if direction == "LONG":
            sl = entry_price - label_cfg.sl_multiplier * atr
        else:
            sl = entry_price + label_cfg.sl_multiplier * atr
        p_k = confidence
        kelly = max((p_k * kelly_b - (1 - p_k)) / kelly_b, 0)
        fwd_pos_size = min(kelly * kelly_fraction, kelly_max)
        if fwd_pos_size < 1e-6:
            continue
        fwd_active = TradeState(
            direction=direction, entry_price=entry_price,
            current_stop_loss=sl, take_profit=0.0, atr_at_entry=atr,
        )
        fwd_entry_bar_local = i
        fwd_daily_count += 1
        continue

    # Only generate signals after cutoff
    if i < fwd_test_start_idx:
        fwd_skip['warmup'] += 1
        continue

    if i >= len(fwd_probs):
        continue

    p = fwd_probs[i]
    cls = int(np.argmax(p))
    if cls == 1:
        fwd_skip['hold'] += 1
        continue
    if max(p[0], p[2]) < exec_cfg.confidence_threshold:
        fwd_skip['low_conf'] += 1
        continue
    if fwd_daily_count >= exec_cfg.max_trades_per_day:
        fwd_skip['daily_cap'] += 1
        continue

    atr = row['atr_14'] if 'atr_14' in df_fwd_bt.columns else 0.0
    if atr < 1e-10:
        fwd_skip['no_atr'] += 1
        continue
    adx_val = row['adx_14'] if 'adx_14' in df_fwd_bt.columns else 999.0
    if adx_val < MIN_ADX:
        fwd_skip['low_adx'] += 1
        continue
    atr_pct = row['atr_pctile'] if 'atr_pctile' in df_fwd_bt.columns else 1.0
    if atr_pct < MIN_ATR_PCTILE:
        fwd_skip['low_volatility'] += 1
        continue
    if i < len(fwd_regime) and fwd_regime[i, 2] > CHOPPY_THRESHOLD:
        fwd_skip['choppy'] += 1
        continue
    if i + 1 >= n_fwd_bars:
        fwd_skip['no_next_bar'] += 1
        continue

    fwd_pending = {
        'direction': "LONG" if cls == 2 else "SHORT",
        'atr': atr,
        'confidence': max(p[0], p[2]),
    }

# Force-close
if fwd_active is not None and fwd_entry_bar_local >= fwd_test_start_idx:
    last_row = df_fwd_bt.iloc[-1]
    if fwd_active.direction == "LONG":
        unr = (last_row['close'] - fwd_active.entry_price) / fwd_active.entry_price
    else:
        unr = (fwd_active.entry_price - last_row['close']) / fwd_active.entry_price
    gross = fwd_active.pnl + unr * fwd_active.remaining_size
    slip_bps = fwd_get_slippage_bps(fwd_active.atr_at_entry)
    cost = entry_cost_frac(slip_bps) + exit_cost_frac('CLOSED_FOLD_END', slip_bps)
    unit_net = (gross - cost) * fwd_pos_size
    lev_net = unit_net * LEVERAGE
    fwd_cum_pnl += lev_net
    fwd_trades.append(dict(
        direction=fwd_active.direction, entry_price=fwd_active.entry_price,
        bars_held=fwd_active.bars_held, status='CLOSED_END',
        gross_pnl=gross * fwd_pos_size * LEVERAGE, net_pnl=lev_net,
        cost=cost * fwd_pos_size * LEVERAGE, position_size=fwd_pos_size,
        slippage_bps=slip_bps, timestamp=last_row.name,
    ))
    fwd_equity.append(fwd_cum_pnl)

# ============================================================
# Forward Test Results
# ============================================================
fwd_df = pd.DataFrame(fwd_trades)

if len(fwd_df) == 0:
    print(f"\n*** NO TRADES in forward test period ({FWD_TEST_CUTOFF} → present) ***")
    print(f"Skip reasons: {fwd_skip}")
else:
    fwd_net = fwd_df['net_pnl'].values
    fwd_n = len(fwd_df)
    fwd_wins = fwd_net[fwd_net > 0]
    fwd_losses = fwd_net[fwd_net < 0]
    fwd_wr = len(fwd_wins) / fwd_n
    fwd_pf = abs(fwd_wins.sum() / fwd_losses.sum()) if len(fwd_losses) else float('inf')

    fwd_df['date'] = pd.to_datetime(fwd_df['timestamp']).dt.date
    fwd_daily = fwd_df.groupby('date')['net_pnl'].sum()
    fwd_range = pd.date_range(fwd_daily.index.min(), fwd_daily.index.max(), freq='D')
    fwd_daily = fwd_daily.reindex(fwd_range, fill_value=0.0)
    fwd_sharpe = fwd_daily.mean() / (fwd_daily.std() + 1e-10) * np.sqrt(365)
    fwd_cum_arr = np.cumsum(fwd_daily.values)
    fwd_peak = np.maximum.accumulate(fwd_cum_arr)
    fwd_dd = fwd_peak - fwd_cum_arr
    fwd_mdd = fwd_dd.max()

    n_days = (fwd_daily.index[-1] - fwd_daily.index[0]).days + 1

    print()
    print('=' * 60)
    print(f'       FORWARD TEST — {FWD_TEST_CUTOFF} → Present (Fold {last_fold_idx})')
    print(f'       (Leverage: {LEVERAGE}x, {n_days} days)')
    print('=' * 60)
    print(f'  Total trades       : {fwd_n}')
    print(f'  Win rate           : {fwd_wr:.4f} ({fwd_wr*100:.1f}%)')
    print(f'  Profit factor      : {fwd_pf:.4f}')
    print(f'  Expectancy/trade   : {fwd_net.mean()*100:.4f}%')
    print(f'  Daily Sharpe       : {fwd_sharpe:.4f}')
    print(f'  Max Drawdown       : {fwd_mdd*100:.4f}%')
    print(f'  Net PnL            : {fwd_net.sum()*100:.2f}%')
    print()
    print(f'  Skip reasons: {fwd_skip}')
    print()

    # Interpretation
    print('-' * 60)
    print('  INTERPRETATION')
    print('-' * 60)
    if fwd_wr > 0.60 and fwd_pf > 2.0:
        print('  STRONG: Model performs well on truly unseen recent data.')
        print('  Good candidate for live deployment.')
    elif fwd_wr > 0.50 and fwd_pf > 1.2:
        print('  MODERATE: Edge exists on recent data but with degradation.')
        print('  Consider paper trading before live.')
    elif fwd_wr > 0.40 and fwd_pf > 1.0:
        print('  WEAK: Marginal edge. May not survive live trading costs.')
    else:
        print('  NEGATIVE: Model fails on recent data.')
        print('  DO NOT deploy live. Retrain or investigate.')

    # Comparison table
    print()
    print('-' * 60)
    print('  COMPARISON: In-Sample vs OOS 2020 vs Forward')
    print('-' * 60)
    print(f'  {"Metric":<20s} {"In-Sample":>12s} {"OOS 2020":>12s} {"Forward":>12s}')
    print(f'  {"Win Rate":<20s} {"70.3%":>12s} {"52.6%":>12s} {f"{fwd_wr*100:.1f}%":>12s}')
    print(f'  {"Profit Factor":<20s} {"3.53":>12s} {"1.16":>12s} {f"{fwd_pf:.2f}":>12s}')
    print(f'  {"Daily Sharpe":<20s} {"10.51":>12s} {"1.32":>12s} {f"{fwd_sharpe:.2f}":>12s}')
    print(f'  {"MDD":<20s} {"0.87%":>12s} {"3.78%":>12s} {f"{fwd_mdd*100:.2f}%":>12s}')
    print('=' * 60)

    # Weekly breakdown
    fwd_df['week'] = pd.to_datetime(fwd_df['timestamp']).dt.to_period('W')
    fwd_weekly = fwd_df.groupby('week')['net_pnl'].agg(['sum', 'count'])
    fwd_weekly.columns = ['return_pct', 'trades']
    fwd_weekly['return_pct'] *= 100
    print('\nForward Test Weekly Returns:')
    print(fwd_weekly.to_string())

    # Chart
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    fwd_cum_pct = np.cumsum(fwd_daily.values) * 100
    axes[0].plot(fwd_daily.index, fwd_cum_pct, linewidth=1.5, color='#4CAF50')
    axes[0].fill_between(fwd_daily.index, 0, fwd_cum_pct, alpha=0.1, color='#4CAF50')
    axes[0].axhline(0, color='grey', linestyle='--', alpha=0.5)
    axes[0].set_title(f'Forward Test Equity ({LEVERAGE}x)')
    axes[0].set_ylabel('Cumulative PnL (%)')
    axes[0].grid(True, alpha=0.3)

    axes[1].fill_between(fwd_daily.index, 0, -fwd_dd * 100, alpha=0.4, color='red')
    axes[1].set_title('Forward Test Drawdown')
    axes[1].set_ylabel('Drawdown (%)')
    axes[1].grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()